# Scrapers for MyDramaList

You will follow the instructions in Part 4 of Week 2 Tasks. In the top part of the notebook summarize through a table of content what you decided to do and then explain why.

**Author:** Kelly Chen\
**Date:** 9/15/26

**Table of Contents:**
1. [Research Question](#sec1)
2. [Scraping Aggregated Top Dramas](#sec2)
3. [Scraping Drama Page Details ](#sec3)

<a id="sec1"></a>

## 1. Research Question

I am interested in a variety of different genres of shows, ranging from historical fantasy and romance to thriller. As many of my favorite shows are of different genres, I would like to see how this correlates to dramas on MyDramaList.

Therefore, my research question looks into what **genres** dominate or are most common in the **top dramas** ranked on MyDramaList.

To do this, I will first scrape the top dramas on MyDramaList, extrating title, rank, type (Korean, Chinese, etc.), genre, year, rating, and number of episodes. I will focus on the genre and rank to answer my research question. One more interesting thing to explore and add to my research question is how the genres for **each type** of drama on the top-ranked list vary.

<a id="sec2"></a>

## 2. Scraping Aggregated Top Dramas 

In [4]:
import requests
from bs4 import BeautifulSoup
from seleniumbase import Driver
import math
import re
import json

#### Try Using Selenium first

Add the parse_drama() function used previously here and modify as needed for scraping from MyDramaList:

In [91]:
def sel_parse_drama(card):
    """Parses HTML content and returns information on drama card on each page"""
    soup = BeautifulSoup(card.get_attribute("outerHTML"), "html.parser")

    ranking = soup.find("div", class_="ranking").get_text(strip=True)
    title = soup.find("h6", class_="title").get_text(strip=True)

    meta = soup.find("span", class_="text-muted").get_text(strip=True)
    type, year, eps = re.findall(r"(.+?) - (\d{4}), (.+)", meta)[0] #use regex

    rating = soup.find("span", class_="score").get_text(strip=True)

    return {
        "rank": ranking,
        "title": title,
        "year": year,
        "eps": eps,
        "rating": rating,
        "type": type
    }

In [92]:
url = "https://mydramalist.com/shows/top"

drama_data = []

with Driver(browser="firefox") as driver:
    driver.open(url)

    # 1. Extract total results count 
    total_text = driver.get_text("p.m-b-sm.pull-right") # This will be a string value
    total_results = int(total_text.split()[0])
    #print(total_results) #5000 results

    # 2. Calculate total pages (20 items per page)
    total_pages = math.ceil(total_results / 20)
    print(f"Total results: {total_results} | Total pages: {total_pages}")

    # 4. Loop through each page URL
    for page in range(1, total_pages + 1):
        driver.open(f"{url}?page={page}")
        driver.sleep(1.0)

        # 5. Extract items on the current page
        #cards = driver.find_elements(".box") # found 23 but only 20 dramas; need to get drama cards only
    
        container = driver.find_element("div.m-t.nav-active-border.b-primary")
        cards = container.find_elements("css selector", ".box")
        current_count = len(cards)

        print(f"Page {page}: {len(cards)} dramas")

        # 6. extract info with parse_drama()
        for card in cards:
            drama = sel_parse_drama(card)
            drama_data.append(drama)

with open("top_dramas_selenium.json", "w") as f:
   json.dump(drama_data, f, indent=4)

Total results: 5000 | Total pages: 250
Page 1: 20 dramas
Page 2: 20 dramas
Page 3: 20 dramas
Page 4: 20 dramas
Page 5: 20 dramas
Page 6: 20 dramas
Page 7: 20 dramas
Page 8: 20 dramas
Page 9: 20 dramas
Page 10: 20 dramas
Page 11: 20 dramas
Page 12: 20 dramas
Page 13: 20 dramas
Page 14: 20 dramas
Page 15: 20 dramas
Page 16: 20 dramas
Page 17: 20 dramas
Page 18: 20 dramas
Page 19: 20 dramas
Page 20: 20 dramas
Page 21: 20 dramas
Page 22: 20 dramas
Page 23: 20 dramas
Page 24: 20 dramas
Page 25: 20 dramas
Page 26: 20 dramas
Page 27: 20 dramas
Page 28: 20 dramas
Page 29: 20 dramas
Page 30: 20 dramas
Page 31: 20 dramas
Page 32: 20 dramas
Page 33: 20 dramas
Page 34: 20 dramas
Page 35: 20 dramas
Page 36: 20 dramas
Page 37: 20 dramas
Page 38: 20 dramas
Page 39: 20 dramas
Page 40: 20 dramas
Page 41: 20 dramas
Page 42: 20 dramas
Page 43: 20 dramas
Page 44: 20 dramas
Page 45: 20 dramas
Page 46: 20 dramas
Page 47: 20 dramas
Page 48: 20 dramas
Page 49: 20 dramas
Page 50: 20 dramas
Page 51: 20 dramas
P

**Fixed** 
At first, selenium could open the website but it seemed to time out; selenium can open the url but cannot parse the content because it could not find the object I was trying to scrape. Therefore tried scraping with BeautifulSoup instead.

#### Try BeautifulSoup to get HTML content:

In [11]:
def fetch_page_content(url):
    """Fetches HTML content from a URL and checks status code."""
    response = requests.get(url)

    print(f"URL: {url}")
    print(f"Status Code: {response.status_code}")

    if response.status_code == 200:
        return response.text
    else:
        print(f"Failed to fetch page. Status code: {response.status_code}")
        return None

In [96]:
def parse_drama(html_content):
    """Parses HTML content and returns information on dramas as a list of dictionaries"""
    if not html_content:
        return []

    soup = BeautifulSoup(html_content, "html.parser") 
    
    container = soup.find("div", class_="m-t nav-active-border b-primary")
    cards = container.find_all("div", class_="box")
    #print(len(cards)) 20 cards on each page; 5000 total

    drama_data = []

    for card in cards:
        title = card.find("h6", class_="title").get_text(strip=True)
      
        ranking = card.find("div", class_="ranking").get_text(strip=True)
        
        meta = card.find("span", class_="text-muted").get_text(strip=True)
        #type, year, eps = re.findall(r"(^\w+? Drama) - (\d{4}), (\d+)", meta)[0] #use regex and get first instance
        
        rating = card.find("span", class_="score").get_text(strip=True)
        
        drama = {
            "title": title,
            "rank": ranking,
            "meta": meta,
            #"year": year,
            #"eps": eps,
            "rating": rating,
            "type": type
        }
        drama_data.append(drama)
    return drama_data
   

In [97]:
#page = fetch_page_content(url)
#print(page)

all_dramas = []

for page_num in range(1, 251):
    url = f"https://mydramalist.com/shows/top?page={page_num}"
    
    page = fetch_page_content(url)
    dramas = parse_drama(page)
    
    all_dramas.extend(dramas) #returns one list instead of lists of lists with append

print(len(all_dramas))

URL: https://mydramalist.com/shows/top?page=1
Status Code: 200
URL: https://mydramalist.com/shows/top?page=2
Status Code: 200
URL: https://mydramalist.com/shows/top?page=3
Status Code: 200
URL: https://mydramalist.com/shows/top?page=4
Status Code: 200
URL: https://mydramalist.com/shows/top?page=5
Status Code: 200
URL: https://mydramalist.com/shows/top?page=6
Status Code: 200
URL: https://mydramalist.com/shows/top?page=7
Status Code: 200
URL: https://mydramalist.com/shows/top?page=8
Status Code: 200
URL: https://mydramalist.com/shows/top?page=9
Status Code: 200
URL: https://mydramalist.com/shows/top?page=10
Status Code: 200
URL: https://mydramalist.com/shows/top?page=11
Status Code: 200
URL: https://mydramalist.com/shows/top?page=12
Status Code: 200
URL: https://mydramalist.com/shows/top?page=13
Status Code: 200
URL: https://mydramalist.com/shows/top?page=14
Status Code: 200
URL: https://mydramalist.com/shows/top?page=15
Status Code: 200
URL: https://mydramalist.com/shows/top?page=16
St

Save into a csv file:

In [93]:
import csv

with open("top_dramas_bs.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=all_dramas[0].keys())
    writer.writeheader()
    writer.writerows(all_dramas)

<a id="sec3"></a>

## 3. Scraping Drama Page Details

Write a function for scraping drama details:

In [157]:
def parse_drama_details(html_content):
    """Parses HTML content and returns drama details as a dictionary"""
    if not html_content:
        return {}

    soup = BeautifulSoup(html_content, "html.parser") 
    
    details = soup.find("div", class_="col-sm-8")
    #print(details)
    
    rating = details.find("div", class_="col-film-rating").get_text(strip=True) #number rating only

    # aggregate rating
    hfs = details.find_all("div", class_="hfs")
    ratings = hfs[0].get_text(" ", strip=True) # add space between separate elements
    watchers = hfs[1].get_text(" ", strip=True)
    reviews = hfs[2].get_text(" ", strip=True)

    synopsis = details.find("div", class_="show-synopsis").find("p").get_text(strip=True)

    #lists
    lists = details.find_all("li", class_="list-item p-a-0")
    #print(lists)
    native_title = lists[0].get_text(" ", strip=True)
    other_names = lists[1].get_text(" ", strip=True)
    screenwriter = lists[2].get_text(" ", strip=True)
    director = lists[3].get_text(" ", strip=True)

    genres = details.find("li", class_="list-item p-a-0 show-genres").get_text(" ", strip=True)
    
    tags = details.find("li", class_="list-item p-a-0 show-tags").get_text(" ", strip=True)

    drama_details = {
        "rating": rating,
        "aggregate_ratings": ratings,
        "watchers": watchers,
        "reviews": reviews,
        "synopsis": synopsis,
        "native_title": native_title,
        "other_names": other_names,
        "screenwriter": screenwriter,
        "director": director,
        "genres": genres,
        "tags": tags
    }

    return drama_details


In [158]:
url = "https://mydramalist.com/760409-zhu-yu" #page for Pursuit of Jade

page = fetch_page_content(url)
details = parse_drama_details(page)
for key, value in details.items():
    print(f"{key}: {value}")

URL: https://mydramalist.com/760409-zhu-yu
Status Code: 200
rating: 9.1
aggregate_ratings: Ratings: 9.1 /10 from 42,737 users
watchers: # of Watchers: 82,629
reviews: Reviews: 656 users
synopsis: It follows Fan Chang Yu, a butcher’s daughter, and Xie Zheng, a fallen noble seeking revenge. Their fake marriage turns into true love, but war tears them apart. Determined, Fan Chang Yu wields her butcher’s knife on the battlefield, searching for justice and her husband. Meanwhile, Xie Zheng reclaims his title, fighting to protect his country and love. Reunited in battle, they stand together, defying fate and uncovering the truth.

(Source: WeTV)~~ Adapted from the web novel "Zhu Yu" (逐玉) by Tuan Zi Lai Xi (团子来袭).Edit Translation
native_title: Native Title: 逐玉
other_names: Also Known As: Chasing Jade ,  Zhu Yu
screenwriter: Screenwriter: Zou Yue
director: Director: Zeng Qing Jie
genres: Genres: Historical , Mystery , Romance , War
tags: Tags: Fake To Real Lovers , Physically Strong Female Lea